In [177]:
#.venv 
prod_connection_string = "DRIVER={ODBC Driver 17 for SQL Server};Server=CUBO-INTERMODA;Database=IMClientesIV;UID=iditm;PWD=Int3r-M0d@.Id@;Trusted_Connection=no;"
url = "https://unikfashiongt.odoo.com"
db = "rocketgithub-unikfashiongt-odoo-sh-main-25251833"
username = "rmartinez@intermoda.com.hn"
password = "Intermod@2026/?"

#Autenticación con Odoo
from datetime import datetime
import xmlrpc.client
import pandas as pd
import pyodbc
import json


common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid = common.authenticate(db, username, password, {})

In [179]:
#Obtener el producto por su código de barras
models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object")
products = pd.DataFrame(models.execute_kw(db, uid, password, 'product.product', 'search_read', [[['barcode', '!=', False]]] ,{'limit': 1})).rename(columns={'barcode': 'CodigoBarra'})
#print(products)

In [180]:
#obtener los clientes
with pyodbc.connect(prod_connection_string) as conn:
    cursor = conn.cursor()
    cursor.execute("EXEC dbo.SP_ObtenerClientes")

    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()

    df = pd.DataFrame.from_records(rows, columns=columns)

    conn.commit()
Clientes = df[df['CodigoCliente'] == "IMGT-000001134"]
#print(Clientes)


In [181]:
#obtener las Tiendas
with pyodbc.connect(prod_connection_string) as conn:
    cursor = conn.cursor()
    cursor.execute("EXEC dbo.SP_ObtenerTiendas ?", ("IMGT-000001134",))
    
    columns = [column[0] for column in cursor.description]
    rows = cursor.fetchall()
    
    df = pd.DataFrame.from_records(rows, columns=columns)
    conn.commit()
Tiendas = df
#print(Tiendas)

In [ ]:
#obtener la información del producto por su código de barras

for index, row in products.iterrows():
    with pyodbc.connect(prod_connection_string) as conn:
        cursor = conn.cursor()
        query = "EXEC dbo.SP_GetCodigosDeBarraInfo ?;"
    
        json_data = row[['CodigoBarra']].to_json(orient='index', force_ascii=False)
            
        cursor.execute(query, json_data)
        columns = [column[0]  for column in cursor.description]
        rows = cursor.fetchall()
        results = pd.DataFrame.from_records(rows, columns=columns)

        create_date = datetime.strptime(row["create_date"], "%Y-%m-%d %H:%M:%S")

        #Crear json para enviar a la base de datos de Intermoda
        json_data = {
            "CodigoCliente": Clientes['CodigoCliente'].values[0],
            "Cliente": Clientes['Cliente'].values[0],
            "CodigoTienda": None,
            "Tienda": None,#Validar de donde viene la tienda con la gente de odoo
            "FechaCreacion": create_date.strftime("%Y-%m-%d"),
            "año": create_date.year,
            "NoMes": create_date.month,
            "Mes": create_date.strftime("%B"),
            "dia": create_date.day,
            "CodigoBarra": row['CodigoBarra'],
            "CodigoArticulo": results['CodigoArticulo'].values[0],
            "Descripcion": results['Descripcion'].values[0],
            "CodigoColor": results['CodigoColor'].values[0],
            "Color": results['Color'].values[0],
            "Talla": results['Talla'].values[0],
            "linea": results['Linea'].values[0],
            "Sublinea": results['Sublinea'].values[0],
            "Categoria": results['Categoria'].values[0],
            "Base": results['Base'].values[0],
            "Genero": results['Genero'].values[0],
            "Clasificacion": results['ClasificacionAX'].values[0],
            "LoteOrigen": results['LoteOrigen'].values[0],
            "PedidoVenta": None,
            "FechaFactura": None,
            "Costo": None,
            "Precio": row['list_price'],
            "Cantidad": row['qty_available'],
            "CostoTotal": None,
        }        

        print(json_data)

        #Enviar json a la base de datos de Intermoda

        #obtener las ventas 
        


{'CodigoCliente': 'IMGT-000001134', 'Cliente': 'UNIK FASHION, SOCIEDAD ANONIMA', 'CodigoTienda': None, 'Tienda': None, 'FechaCreacion': '2026-03-19', 'año': 2026, 'NoMes': 3, 'Mes': 'March', 'dia': 19, 'CodigoBarra': '7424625714570', 'CodigoArticulo': '10 11 01 03 897 0001', 'Descripcion': 'BERMUDA HOMBRE STRAIGHT MEDIUM RISE', 'CodigoColor': 'T2', 'Color': 'INDIGO MEDIO', 'Talla': '29', 'linea': 'DENIM', 'Sublinea': 'DENIM', 'Categoria': 'MODA', 'Base': 'VI897', 'Genero': 'Hombre', 'Clasificacion': 'PRIMERAS', 'LoteOrigen': '622F', 'PedidoVenta': None, 'FechaFactura': None, 'Costo': None, 'Precio': 265.0, 'Cantidad': 1.0, 'CostoTotal': None}
